In [6]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")   # a harmless warning from LAL
from gwpy.timeseries import TimeSeries
import matplotlib.pyplot as plt
import numpy as np
from gwpy.timeseries import TimeSeries
from gwpy.io.gwf import iter_channel_names

from pycbc.waveform import get_td_waveform
import pylab
from scipy.signal import welch
from pycbc.types import FrequencySeries

from pycbc.waveform import get_fd_waveform
from pycbc.filter import matched_filter
from scipy.signal import windows
from scipy.signal import iirnotch, filtfilt
from gwosc.datasets import event_gps


# Challenge 1: SNR Timeseries and Coalesence Time

In this challeng we are analyzing real LIGO data from the Hanford detector that detected the event GW170814 https://arxiv.org/pdf/1709.09660. 

**Task**: 
- Calculate a SNR time series using a template waveform with the masses of the binary as stated in the paper. 
- What is the coalesence time of that binary? 
- What SNR does this source have in the Hanford detector? 

Roughly follow the template below to provide your answer. You can crosscheck your results with the findings in the paper. 
Feel free to add cells whereever you like. We encourage you to make plots of the quantities you are investigating!

In [7]:
tevent = event_gps("GW170814")
segment = (int(tevent) - 16, int(tevent) + 16)
h1_ts = TimeSeries.fetch_open_data("H1", *segment, sample_rate=4096, cache=True, verbose=True)

gpstime = h1_ts.times.value
relative_times = gpstime - tevent

h1_np = np.array(h1_ts)
dt = h1_ts.dt.value
Tobs = h1_ts.times.value[-1]-h1_ts.times.value[0]

Fetched 1 URLs from gwosc.org for [1186741845 .. 1186741877))
Reading data... [Done]


## Functions

The noise weighted inner product between two time series a and b is defined as $$(a|b) \equiv 4\, \mathrm{Re} \int_{0}^{\infty} \frac{\tilde{a}(f)\,\tilde{b}^{*}(f)}{S_n(f)}\, df $$
The matched filter SNR of a signal s with a template h is defined as  $$ \rho = \frac{(s|h)}{\sqrt{(h|h)}}$$
Calculating this SNR between the signal $s(t)$ and a time shifted template $h(t-\Delta t)$ leaves us with a SNR time series.$$ \frac{(s|h(t - \Delta t))}{\sqrt{(h|h)}} = \frac{4}{\sqrt{(h|h)}}\, \mathrm{Re} \int_{0}^{\infty} \frac{\tilde{s}(f)\,\tilde{h}^{*}(f)\, e^{2\pi i f \Delta t}}{S_n(f)} \, df ..
$$ The SNR peak indicates the best line up between template and signal in time domain and thus tells us the most likely coalescence time given the template parameters and assuming a merger time of the template at t = 0. 
Make sure you use the non whitened verison so that this version of the inner product is correct! 

Notice that the above time shifted inner product looks like a fourier transform itsself! Use this in your calculation of the SNR for each time shift in order to speed up the calculation using np.fft.ifft(). Think about the normalization that is assumed here! https://numpy.org/doc/2.2/reference/routines.fft.html#module-numpy.fft

In [3]:
## --- TO DO:
def matched_filter_fft(data_fd,template_fd,freq,psd):
    """
    For simplicity, we suggest that you use dt as the time difference between shifts and Tobs as the maximum time shift.
    function input:
    data_fd     : frequency domain data 
    template_fd : frequency domain template on the same frequencies
    freq        : frequencies of data_fd / template_fd / psd 
    psd         : psd of the noise at freq
    """
    
    
    # your code here
    return ...

## The Data

Transform the numpy version of the time domain data to frequency domain using the fft. 

In [4]:
# ---- TO DO: tranform the data to frequency domain - remember that you always need to window! ----
win = windows.tukey(...)  

h1_windowed = h1_np * win                    
h1_fs = ... # the frrequency domain data

freqs_values = ...
df = ...

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'ellipsis'

We are not interested in all of the frequencies in our data. Only keep the data in the frequency window [20 Hz, 500 Hz]

## The PSD

Get the noise PSD from the offsource data. To do this, plot the time domain GW strain and identify the signals in the data using whitening and bandpassing techniques you learned about in the earlier Lab sessions. Decide on which part of the data to use to estimate the PSD of the detector noise. 

In [ ]:
# ----- TO DO: get the noise psd from the data -- 

psd_freqs, psd_welch = welch(...)

# --- interpolate the psd so that it matched the frequency grid of the frequency domain data
psd = ...

## The waveform

Generate the GW waveform directly in frequency domain using the IMRPhenomD model. 

In [ ]:
# ---- TO DO: get the frequency domain template - generate it on the frequency grid of the data

#source parameters
m_1 = ...
m_2 = ...
distance = ...
ra = 0.83
dec = 0.77 

d = Detector("H1")
Fp, Fc = d.antenna_pattern(ra, dec, 0, gpstime[0])

hp_fd, hc_fd = get_fd_waveform(approximant="IMRPhenomD", ...) #you can set f_low = 20 Hz, it still starts the waveform at 0, below 20 Hz it just sets it to zero
h_fd = hp_fd * Fp + hc_fd * Fc 

template_fd = np.zeros(len(freqs_values), dtype=complex)
template_fd = h_fd

## Compute the SNR timeseries

For simplicity, we suggest that you use **dt** as the time difference between shifts and **Tobs** as the maximum time shift. 

In [ ]:
# --- TO DO: Implement the answers to the questions in the Task description here

time_shifts, z = matched_filter_fft(h1_fs, template_fd, freqs_values, psd)
rho = np.abs(z)